## Terms

- Lot: spaces for a component's most atomic entity to be placed (synonymous with "Homes" in residential parlance)
- Day: 1 iteration
- Month: 28 iterations
- Year: 12 * 28 iterations

## Demand target to demand valve

- All three RCI components' demand moves, moderated by a valve. 
- The demand-target is set monthly (each 28 iterations), and then the demand moves towards it slowly each day.
- Each component has a discrete rule for how it moves towards the target

## Lot increase/decrease

- New lots are introduced when the existing lots are 95% filled
- Existing lots are removed when they're 50% filled
- Number of lots added or removed is calculated weekly (7th iteration)
- 

## Residential pop increase

- Residential demand-target goes up when there's spare jobs available (commercial pop + industrial pop is greater than residential pop). Target is updated once per month.
- There's a "valve" effect, so the actual residential demand trails the demand-target. Trailing happens daily.
- Migration goes up when there's spare residential demand & down on the inverse.
- Residential demand is filled by migration (rate) + births (rate) - deaths (rate). This is applied daily. Capped by the number of available homes.
- A count of new homes between `0 to min(required, cap)` are added in each weekly iteration
- Residents are paid $1000 per month (if they're employed?)
- Residents consume daily

### Residential utility
- Residents have a marginal utility calculation, and consume as utility-maximising actors
- Residents consume from commercial output
- If the residents utility is unmet, they will migrate out (this isn't currenty implemented, residents leave when unemployed only at the moment).

## Commercial pop increase
- Commercial demand target is computed monthly from weighted normalised signal, bound to -100,100.
- Demand valve trails the target
- Commercial expansion happens when residents are underserved (either subsistence shortfall, or discretionary spending shortfall). Demand is stronger if the shortfall is subsistence (because it indicates there's starvation and Residnets will move out).
- Commercial contraction happens when commercial capacity is underutilised (there's units left over in the cohort's stock at the end of each resident-day)

- Commercial demand-target goes up when residents marginal utility is under-maximised. The sim produces `unfilled_discretionary_demand` daily, and the demand-target is moved monthly depending on that value. Calcualted monthly (28d)
- New commercial outlets open at a migration-rate towards `demand_target`
- Residents work at commercial outlets, and are paid by it.
- Commercial outlets make profit and loss, and if they make a loss, they close.
- A count of new lots between `0 to min(required, cap)` are added in each weekly iteration


### Commercial profit maximisation

- Commercial outlets are profit maximising entities
- If they run out of money, they go out of business, and their employees do not get paid
- Commercial entities can purchase `n * num_employees` 

## Industrial pop increase
- All commercials are consumers of industry
- Industrial demand-target goes up when commercial 
- Industry buys from a global unlimited wholesaler, the wholesaler price fluctuates +/- 10%

### Industrial profit maximising
- Industrial outlets are profit maximising entities
- If thety run out of money, they go out of business, and their employees do not get paid

## Employment (mostly not implemented)

- Residents work either in Commercial or Industrial entities
- Comm/Ind entities have an employee cap defined by `X_Entity.jobs`
- For commercial entities, if they finish the turn with excess cash, they have an `n%` chance to increase their `.jobs` capacity by 1
- Each additional employee allows the entity to carry additional stock (`stock_capacity * geometric_decay_factor`)
- If the employer doesn't have the cash at the end of the turn to meet their employment commitments, the employer's `.jobs` count falls by a min of 10% or 1 `min(.jobs * 0.1, 1)`
- If the employer's `.jobs` value ever falls to 0, and they have no stock, they are bankrupt


## Bankruptcy (partly implemented)

- Commercial & industrial can go bankrupt if:
  - They have 0 `.jobs` capacity remaining
  - They haven't got enough money to purchase the cheapest good from their supply chain
- Any remaining stock is returned to the wholesaler



In [128]:
from typing import Tuple
import math
from decimal import Decimal
import enum
import datetime
import uuid

from IPython.display import display
from utils.plotly_graphs.demand_graphs import plot_demand_detail, plot_demand_history, plot_jobs_history
from utils.plotly_graphs.consumer_utility_graphs import plot_welfare_history, plot_throughput_history, plot_shortage_vs_demand_target


history = {
    'residential_population': [],
    'commercial_population': [],
    'industrial_population': [],
    'residential_demand': [],
    'commercial_demand': [],
    'industrial_demand': [],
    'residential_demand_target': [],
    'commercial_demand_target': [],
    'industrial_demand_target': [],
    'residential_lots': [],
    'commercial_lots': [],
    'industrial_lots': [],
    'jobs_per_resident': [],
    'migration': [],
    'deaths': [],
    'births': [],
    'unemployment_rate': [],
    'vacancy_rate': [],
    'resident_subsistence_unmet': [],
    'subsistence_shortfall_units': [],
    'units_sold': [],
    'unfilled_discretionary_demand': [],
    'commercial_subsistence_unmet': [],
    'units_sold_ind_to_comm': [],
    'total_money_in_economy': [],
    'total_money_in_industrial': [],
    'total_money_in_commercial': [],
    'total_money_in_residential': [],
    'total_money_in_wholesale': [],
    'total_money_in_government': []
}

In [129]:
class GlobalConfig:
    def __init__(self):
        self.GLOBAL_residential_lots_cap = 10000
        self.GLOBAL_commercial_lots_cap = 5000
        self.GLOBAL_industrial_lots_cap = 5000
        self.GLOBAL_jobs_per_commercial_pop = 1 # TODO: This is not implemented in the CommercialEntity class yet
        self.GLOBAL_jobs_per_industrial_pop = 1 # TODO: This is not implemented in the IndustrialEntity class yet
        self.GLOBAL_birth_rate = 0 #0.01
        self.GLOBAL_death_rate = 0 #0.005

        self.GLOBAL_residential_starting_money = Decimal(5000)
        self.GLOBAL_commercial_starting_money = Decimal(5000)
        self.GLOBAL_industrial_starting_money = Decimal(10000)

        self.GLOBAL_days_per_week = 7
        self.GLOBAL_days_per_month = self.GLOBAL_days_per_week * 4
        self.GLOBAL_days_per_year = self.GLOBAL_days_per_month * 12

        # Units per day of utility the resident requires to 
        # survive. If they don't have this much utility, they'll migrate out
        # in the next day.
        self.GLOBAL_residential_subsistence_quantity = 10

CONF = GlobalConfig()

In [130]:
class ActorEntity:
    name: str
    id_uuid: uuid.UUID
    
    def __init__(self, name: str):
        self.name = name
        self.id_uuid = uuid.uuid4()
        
class CityEntity(ActorEntity):
    def __init__(self):
        super().__init__(name="City")
        self.next_entity_id = 0

class RestOfWorldEntity(ActorEntity):
    def __init__(self, id: int):
        super().__init__(name=f"RestOfWorldEntity_{id}")
        self.id = id

class RciType(enum.Enum):
    RESIDENTIAL = 0
    COMMERCIAL = 1
    INDUSTRIAL = 2


class LotEntity:
    def __init__(self, id: int, lot_type: RciType):
        self.id = id
        self.rci_type: RciType = lot_type
        self.occupied = False
        self.occupied_by: int | None = None


    

class ResidentialEntity(ActorEntity):
    def __init__(self, id: int, lot_address: int):
        super().__init__(name=f"ResidentialEntity_{id}")
        self.id = id
        self.rci_type: RciType = RciType.RESIDENTIAL
        
        self.employed = False
        self.employed_at: int | None = None
        self.employed_type: RciType | None = None

        self.lot_address: int = lot_address

        # Per period consumption and utility tracking vars
        self.lifetime_utility = Decimal(0)
        self.basket_count = 0
        self.period_utility = Decimal(0)
        self.subsistence_q_met = False

        # Base utility is the utility of the initial unit of consumption before decay is applied.
        self.base_utility = Decimal(30.0)
        
        # utility_decay is a geometric decay factor for marginal utility of consumption
        self.utility_decay = Decimal(0.8)

    def marginal_utility(self, units_held: int) -> Decimal:
        """
        Calcualte utility of the next good consumed, based on geometric
        decay from base_utility

        Args:
            - units_held: The number of units already held by the resident.
        Returns:
            - The marginal utility of the next unit of consumption.
        """
        return self.base_utility * (self.utility_decay ** units_held)
    
    def willing_to_buy(self, price: Decimal) -> bool:
        """
        Determine if the resident is willing to buy a good at the given price.

        This follows a quasi-linear utility function, where the resident will
        consume if the next unit's MarginalUtility is greater than or equal to
        the price of the good.

        Args:
            - price: The price of the good to be purchased.
        Returns:
            - True if the resident is willing to buy, False otherwise.
        """
        return self.marginal_utility(self.basket_count) >= price



class CommercialEntity(ActorEntity):
    def __init__(self, id: int, lot_address: int):
        super().__init__(name=f"CommercialEntity_{id}")
        self.id = id
        self.rci_type: RciType = RciType.COMMERCIAL
        self.jobs = 1

        self.lot_address: int = lot_address
        self.stock = 1000
        self.stock_capacity = 1000
        self.stock_price_per_unit = 1.0
        self.employee_wage = 200.0 / 28


class IndustrialEntity(ActorEntity):
    def __init__(self, id: int, lot_address: int):
        super().__init__(name=f"IndustrialEntity_{id}")
        self.id = id
        self.rci_type: RciType = RciType.INDUSTRIAL
        self.jobs = 1
        self.employee_wage = 200.0 / 28
        self.lot_address: int = lot_address

        self.stock = 1000
        self.stock_capacity = 1000
        self.stock_price_per_unit = 0.5
        self.stock_regeneration_rate = 0.1 # 10% of stock capacity per day

class WholesalerEntity(ActorEntity):
    def __init__(self, id: int):
        super().__init__(name=f"WholesalerEntity_{id}")
        self.id = id
        self.rci_type: RciType = RciType.INDUSTRIAL

        self.stock = 1_000_000_000
        self.stock_capacity = 1_000_000_000
        self.stock_price_per_unit = 0.25


## Lots

The cell below contains methods for managing lots. 
    

In [131]:
def add_entity_to_lot(type: RciType, entity_id: int, lots: list[LotEntity]) -> int:
    """
    Add an entity to the city if there's a vacant lot available

    Args:
        - type (RciType): The type of entity to add (residential, commercial, or industrial)
        - entity_id (int): The ID of the entity to add

    Returns:
        - int: The ID of the lot the entity was added to, or -1 if no lot was available

    Raises:
        - ValueError: If there are no vacant lots available for the specified type
    """
    for lot in lots:
        if not lot.occupied and lot.rci_type == type:
            lot.occupied = True
            lot.occupied_by = entity_id
            return lot.id
    return -1

def add_lot(lots: list[LotEntity], type: RciType,
            residential_lots_arr = None,
            commercial_lots_arr = None,
            industrial_lots_arr = None) -> bool:
    """
    Add a lot to the city.
    
    Args:
        - type (RciType): The type of lot to add
    Returns:
        - bool: True if a lot was added
    """
    lot = LotEntity(len(lots), type)
    lots.append(lot)
    {
        RciType.RESIDENTIAL: residential_lots_arr,
        RciType.COMMERCIAL: commercial_lots_arr,
        RciType.INDUSTRIAL: industrial_lots_arr
    }[type].append(lot)
    print(f'Added {type.name.lower()} lot (id: {lot.id})')


def lots_of_type(lots: list[LotEntity], type: RciType) -> list[LotEntity]:
    return [lot for lot in lots if lot.rci_type == type]

## Banking and transactions

The cell below contains structure for the bank, and each entity's account

In [132]:
from exceptions.not_enough_money_error import NotEnoughMoneyError


class TransactionLogEntry:
    timestamp: datetime.datetime
    sender: uuid
    sender_type: str
    receiver: uuid
    receiver_type: str
    for_good_service: str
    amount: Decimal
    sender_balance_after: Decimal
    receiver_balance_after: Decimal

    def __init__(self,
                 sender: uuid,
                 sender_type: str,
                 receiver: uuid,
                 receiver_type: str,
                 for_good_service: str,
                 amount: Decimal,
                 sender_balance_after: Decimal,
                 receiver_balance_after: Decimal):
        self.id = uuid.uuid4()
        self.timestamp = datetime.datetime.now()
        self.sender = sender
        self.sender_type = sender_type
        self.receiver = receiver
        self.receiver_type = receiver_type
        self.for_good_service = for_good_service
        self.amount = amount
        self.sender_balance_after = sender_balance_after
        self.receiver_balance_after = receiver_balance_after



class BankAccount:
    balance: Decimal
    owner_id: str
    owner_type: str

    def __init__(self, owner_id: str, owner_type: str, initial_balance: Decimal = Decimal(0)):
        self.balance = initial_balance
        self.owner_id = owner_id
        self.owner_type = owner_type

class Bank:
    accounts_by_owner: dict[str, BankAccount]
    transaction_log: list[TransactionLogEntry]

    def __init__(self):
        self.accounts_by_owner = {}
        self.transaction_log = []

    def sum_all_accounts(self, entity_type: str | None = None) -> Decimal:
        """
        Calculate the total balance of all bank accounts.

        Returns:
            - Decimal: The sum of balances of all bank accounts.
        """
        total = Decimal(0)
        for account in self.accounts_by_owner.values():
            if entity_type is None or account.owner_type.lower() == entity_type.lower():
                total += account.balance
        return total

    def create_account(self,
                       owner: ActorEntity,
                       initial_balance: Decimal = Decimal(0)) -> BankAccount:
        """
        Create a new bank account for the given owner UUID.

        Args:
            - owner (ActorEntity): The account owner.
        Returns:
            - BankAccount: The newly created bank account for the owner.
        """
        print(f'Creating account for owner {owner.name} (UUID: {owner.id_uuid}, type {type(owner).__name__})')
        if owner.id_uuid in self.accounts_by_owner:
            raise ValueError(f'Account already exists for owner {owner.name} (UUID: {owner.id_uuid})')
        account = BankAccount(owner_id=owner.id_uuid, owner_type=type(owner).__name__, initial_balance=initial_balance)
        self.accounts_by_owner[owner.id_uuid] = account
        return account
    

    def get_balance(self, owner: uuid) -> Decimal:
        """
        Get the balance of a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - Decimal: The balance of the bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        account = self.get_account_by_owner(owner)
        return account.balance
    
    
    def get_account_by_owner(self, owner: uuid) -> BankAccount:
        """
        Get a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        account = self.accounts_by_owner.get(owner)
        if account is not None:
            return account
        raise ValueError(f'No account found for owner {owner}')
    

    def transfer_money(self,
                       sender: uuid,
                       receiver: uuid,
                       amount: Decimal,
                       for_good_service: str) -> None:
        """
        Transfer money from one account to another.

        Args:
            - sender (uuid): The UUID of the sender's account.
            - receiver (uuid): The UUID of the receiver's account.
            - amount (Decimal): The amount of money to transfer.
            - for_good_service (str): One-word description of the good or service this transaction is for (e.g. "orange", "salary", etc.)

        Returns:
            None

        Raises:
            - ValueError: If either the sender or receiver account is not found.
            - NotEnoughMoneyError: If the sender does not have enough money to transfer.
        """
        sender_account = self.get_account_by_owner(sender)
        receiver_account = self.get_account_by_owner(receiver)
        if sender_account.balance < amount:
            raise NotEnoughMoneyError(f'Sender {sender} does not have enough money to transfer {amount}. Current balance: {sender_account.balance}')
        sender_account.balance -= amount
        receiver_account.balance += amount
        transaction_log_entry = TransactionLogEntry(
            sender=sender,
            sender_type=sender_account.owner_type,
            sender_balance_after=sender_account.balance,
            receiver=receiver,
            receiver_type=receiver_account.owner_type,
            receiver_balance_after=receiver_account.balance,
            for_good_service=for_good_service,
            amount=amount
        )
        self.transaction_log.append(transaction_log_entry)

## Clock advance and period procession

The cell below contains functions for moving the clock and periods forwards.

In [133]:
def advance_clock_and_periods(i: int, week: int, day: int, month: int, year: int, conf: GlobalConfig) -> Tuple[bool, bool, bool, int, int, int, int]:
    """
    Advance the clock and periods based on the current iteration.

    Args:
        - i (int): The current iteration count.
        - week (int): The current week.
        - day (int): The current day.
        - month (int): The current month.
        - year (int): The current year.
        - conf (GlobalConfig): The global configuration object.
    Returns:
        - Tuple[bool, bool, bool, int, int, int, int]: A tuple containing flags for:
            - Is the week end,
            - Is the month end,
            - Is the year end,
            - day (in the current week),
            - month (in the current year),
            - year (in the current simulation).
    """
    is_week_end = i % conf.GLOBAL_days_per_week == 0
    is_month_end = i % conf.GLOBAL_days_per_month == 0
    is_year_end = i % conf.GLOBAL_days_per_year == 0

    if is_week_end:
        print(f'It is the end of week {week}\n')
        week += 1
        day = 1
    else:
        day += 1
    if is_month_end:
        print(f'It is the end of month {month_from_int(month)} ({year}).\n')
        month += 1
        week = 1
    if is_year_end:
        print(f'It is the end of year {year}.\n')
        year += 1
        month = 1
    return is_week_end, is_month_end, is_year_end, week, day, month, year

def day_from_int(i: int) -> str:
    days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    return f'{i} ({days[(i - 1) % 7]})'

def month_from_int(i: int) -> str:
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    return f'{i} ({months[(i - 1) % 12]})'

In [134]:




def main(iteration_count = 100):
    the_rest_of_world = RestOfWorldEntity(0)
    the_city = CityEntity()
    the_bank = Bank()

    # create a rest_of_world account with a huge budget
    the_bank.create_account(the_rest_of_world, initial_balance=Decimal(1_000_000_000))
    the_bank.create_account(the_city, initial_balance=Decimal(0))
    the_wholesaler = WholesalerEntity(0)
    the_bank.create_account(the_wholesaler, initial_balance=Decimal(0))

    tax_rate_residential = 0.1
    tax_rate_commercial = 0.15
    tax_rate_industrial = 0.2

    residential_demand_target = 0
    residential_demand = 0
    residential_population = 123
    residential_entities: list[ResidentialEntity] = []

    commercial_demand_target = 0
    commercial_demand = 0
    initial_commercial_population = 180
    commercial_entities: list[CommercialEntity] = []

    industrial_demand_target = 0
    industrial_demand = 0
    initial_industrial_population = 180
    industrial_entities: list[IndustrialEntity] = []

    migration = 0

    lots: list[LotEntity] = []
    residential_lots_arr: list[LotEntity] = []
    commercial_lots_arr: list[LotEntity] = []
    industrial_lots_arr: list[LotEntity] = []

    day = 1
    week = 1
    month = 1
    year = 1950
    total_jobs = 0
    unfilled_jobs = 0
    vacancy_rate = 0
    
    def add_entity_to_city(lots: list[LotEntity], type: RciType, conf: GlobalConfig) -> int:
        """
        Add a new entity to the city.
        
        Create the new entity in the appropriate array, and add it to a lot if one exists.
        If a lot doesn't exist, raises an error.

        Args:
            - type (RciType): The type of entity to add (residential, commercial, or industrial)        
        Returns:
            - int: The ID of the newly added entity
        """
        added_to_lot = add_entity_to_lot(type, the_city.next_entity_id, lots)
        if added_to_lot == -1:
            raise ValueError(f"Failed to add {type.name.lower()} entity (id: {the_city.next_entity_id}) to lot. len(lots[type]) is: {len(lots_of_type(lots, type))}")
        
        created_entity = None
        starting_money = conf.GLOBAL_residential_starting_money if type == RciType.RESIDENTIAL \
            else conf.GLOBAL_commercial_starting_money if type == RciType.COMMERCIAL \
                else conf.GLOBAL_industrial_starting_money

        if type == RciType.RESIDENTIAL:
            created_entity = ResidentialEntity(the_city.next_entity_id, added_to_lot)
            residential_entities.append(created_entity)
        if type == RciType.COMMERCIAL:
            created_entity = CommercialEntity(the_city.next_entity_id, added_to_lot)
            commercial_entities.append(created_entity)
        if type == RciType.INDUSTRIAL:
            created_entity = IndustrialEntity(the_city.next_entity_id, added_to_lot)
            industrial_entities.append(created_entity)
        
        the_bank.create_account(created_entity, initial_balance=Decimal(0))
        the_bank.transfer_money(the_rest_of_world.id_uuid, created_entity.id_uuid, starting_money, for_good_service="initial seed endowment")
        the_city.next_entity_id += 1
        return the_city.next_entity_id - 1

    def remove_employer_entity(entity, entities: list, reason: str) -> dict:
        """
        Remove an employer entity (comm or ind) from the city.

        Frees the entity lot, returns its remaining stock to the wholesaler, and sets
        its employees to unemployed.

        Args:
            - entity (CommercialEntity | IndustrialEntity): The entity to remove from the city.
            - entities (list): The list of entities from which to remove the specified entity (pass in either `commercial_entities` or `industrial_entities`).
            - reason (str): The reason for removing the entity.
        Returns:
            dict: A dictionary containing information about the removed entity, such as its ID and the reason for removal.
        """
        # free the lot
        lot_to_free = next(
            (l for l in lots
             if l.occupied and l.occupied_by == entity.id and l.rci_type == entity.rci_type),
             None,
        )
        if lot_to_free:
            lot_to_free.occupied = False
            lot_to_free.occupied_by = None
        else:
            print(f"Warning: No lot found for entity ID {entity.id} of type {entity.rci_type}.")

        # return remaining stock to wholesaler. Note that the 
        # wholesaler is an unlimited wholesaler, so only the 
        # entity's stock needs adjusting
        stock_to_return = max(entity.stock, 0)
        entity.stock = 0

        remaining_balance = the_bank.get_balance(entity.id_uuid)
        if remaining_balance > 0:
            # transfer the money into the rest_of_world's account
            the_bank.transfer_money(entity.id_uuid, the_rest_of_world.id_uuid, remaining_balance, for_good_service="return remaining balance, entity exiting the city economy")

        # release the employees so that payroll stats are updated for today
        released_employees = 0
        for r in residential_entities:
            if r.employed and r.employed_at == entity.id and r.employed_type == entity.rci_type:
                r.employed = False
                r.employed_at = None
                r.employed_type = None
                released_employees += 1
        entities.remove(entity)
        return {
            'entity_id': entity.id,
            'name': entity.name,
            'rci_type': entity.rci_type,
            'reason': reason,
            'balance_at_removal': the_bank.get_balance(entity.id_uuid),
            'returned_balance': remaining_balance,
            'returned_stock': stock_to_return,
            'released_employees': released_employees,
        }



    def initialize_lots_and_pop_arrays():
        """
        Initialise lots and assign populations to each lot.

        In this model, a resident, commercial, or industrial occupies exactly one lot.
        """
        initial_residential_lots = int(residential_population * 1.35)
        initial_commercial_lots  = int(initial_commercial_population * 1.35)
        initial_industrial_lots  = int(initial_industrial_population * 1.35)

        for _ in range(initial_residential_lots):
            add_lot(lots, RciType.RESIDENTIAL, residential_lots_arr=residential_lots_arr)
        for _ in range(initial_commercial_lots):
            add_lot(lots, RciType.COMMERCIAL, commercial_lots_arr=commercial_lots_arr)
        for _ in range(initial_industrial_lots):
            add_lot(lots, RciType.INDUSTRIAL, industrial_lots_arr=industrial_lots_arr)

        for _ in range(residential_population):
            add_entity_to_city(lots, RciType.RESIDENTIAL, CONF)
        for _ in range(initial_commercial_population):
            add_entity_to_city(lots, RciType.COMMERCIAL, CONF)
        for _ in range(initial_industrial_population):
            add_entity_to_city(lots, RciType.INDUSTRIAL, CONF)


    initialize_lots_and_pop_arrays()
    

    for i in range(1, iteration_count + 1):
        # Phase 1: Advance the clock and periods
        is_week_end, is_month_end, is_year_end, week, day, month, year = advance_clock_and_periods(i, week, day, month, year, CONF)
        print(f'--- ({i}) Day {day_from_int(day)}, Week {week}, Month {month_from_int(month)}, Year {year} ---')
        
        # Phase 2: Compute aggregate values

        def compute_aggregate_values():
            total_jobs = sum(c.jobs for c in commercial_entities) + sum(e.jobs for e in industrial_entities)            
            jobs_gap = total_jobs - residential_population            
            unfilled_jobs = max(0, jobs_gap)
            vacancy_rate = max(0, min(1, unfilled_jobs / total_jobs)) if total_jobs > 0 else 0
            employed_residents = min(residential_population, total_jobs)
            unemployed_residents = residential_population - employed_residents
            unemployment_rate = (unemployed_residents / residential_population) if residential_population > 0 else 0

            # sum the history.units_sold over the last 28 days (1 month)
            print(f'Units sold in the last 28 days: {sum(history["units_sold"][-28:]) if len(history["units_sold"]) >= 28 else sum(history["units_sold"])}')
            monthly_units_sold = sum(history['units_sold'][-28:]) if len(history['units_sold']) >= 28 else sum(history['units_sold'])

            print(f'Unfilled discretionary demand in the last 28 days: {sum(history["unfilled_discretionary_demand"][-28:]) if len(history["unfilled_discretionary_demand"]) >= 28 else sum(history["unfilled_discretionary_demand"])}')
            monthly_unfilled_discretionary_demand = sum(history['unfilled_discretionary_demand'][-28:]) if len(history['unfilled_discretionary_demand']) >= 28 else sum(history['unfilled_discretionary_demand'])

            print(f'Subsistence shortfall units in the last 28 days: {sum(history["subsistence_shortfall_units"][-28:]) if len(history["subsistence_shortfall_units"]) >= 28 else sum(history["subsistence_shortfall_units"])}')
            monthly_subsistence_shortfall_units = sum(history['subsistence_shortfall_units'][-28:]) if len(history['subsistence_shortfall_units']) >= 28 else sum(history['subsistence_shortfall_units'])


            # TODO: record this as a count, instead of a list, and then sum it here.
            # print('Residence Subsistence Unmet History:', history['resident_subsistence_unmet'])
            # print(f'Resident subsistence unmet in the last 28 days: {sum(history["resident_subsistence_unmet"][-28:]) if len(history["resident_subsistence_unmet"]) >= 28 else sum(history["resident_subsistence_unmet"])}')
            # monthly_resident_subsistence_unmet = sum(history['resident_subsistence_unmet'][-28:]) if len(history['resident_subsistence_unmet']) >= 28 else sum(history['resident_subsistence_unmet'])


            print(f'Total Jobs: {total_jobs} for {residential_population} residents, Total Available Jobs: {unfilled_jobs}, Vacancy Rate: {vacancy_rate:.2%}, Unemployment Rate: {unemployment_rate:.2%}')
            return total_jobs, unfilled_jobs, vacancy_rate, employed_residents, unemployed_residents, unemployment_rate, jobs_gap, \
                    monthly_units_sold, monthly_unfilled_discretionary_demand, monthly_subsistence_shortfall_units
        
        total_jobs, \
            unfilled_jobs, \
            vacancy_rate, \
            employed_residents, \
            unemployed_residents, \
            unemployment_rate, \
            jobs_gap, \
            monthly_units_sold, \
            monthly_unfilled_discretionary_demand, \
            monthly_subsistence_shortfall_units = compute_aggregate_values()

        # Phase 3: Decide new targets for demand based on agg vals
        # Phase 3a: Residential demand target
        if is_month_end:
            residential_demand_target = max(-100, min(100, jobs_gap / 4))  # tune divisor
            if residential_demand_target < -100 or residential_demand_target > 100:
                raise ValueError(f"Residential demand target out of bounds: {residential_demand_target}")
            print(f'Total Jobs: {total_jobs} for {residential_population} residents, Total Available Jobs: {unfilled_jobs}')
            print(f'There are {unfilled_jobs} available jobs. Residential demand target will {"increase" if unfilled_jobs > 0 else "decrease"} to {residential_demand_target:.2f}')
        
        residential_demand_step = residential_demand_target - residential_demand
        residential_demand_rise_rate = 1
        residential_demand_fall_rate = 1

        # Phase 3b: Commercial demand target
        if is_month_end:
            normalisation_factor = residential_population * 28 # normalisation factor to per-resident-day
            if normalisation_factor <= 0:
                # handle the case where residential_population is zero (on a city start)
                print('WARN: Attempted to calculate commercial demand target with zero residential population. Setting commercial_demand_target to 0.')
                commercial_demand_target = 0
            else:
                # upward pressure calculations
                
                subsistenace_normalised = monthly_subsistence_shortfall_units / normalisation_factor
                discretionary_normalised = monthly_unfilled_discretionary_demand / normalisation_factor

                # downward pressure calculations
                commercial_capacity_in_units = sum(c.stock_capacity for c in commercial_entities)
                if commercial_capacity_in_units > 0:
                    utilisation = monthly_units_sold / (commercial_capacity_in_units * 28)  # normalise to per-day
                else:
                    utilisation = 0
                TARGET_UTILISATION = 0.85
                slack = max(0, TARGET_UTILISATION - utilisation)

                # Tunable weights for factors affecting the commercial demand target.
                # Discretionary is the baseline weight, and the other factors are weighted
                # relative to it.
                WEIGHT_DISCRETIONARY = 1.0
                WEIGHT_SUBSISTENANCE = 3.0 # subsistence is heavily weighted, becauase it means people are starving.
                WEIGHT_SLACK = 2.0 # slack is moderately highly weighted, because it means the commercial sector is underutilised and can take on more demand.
                GAIN = 4.0 
                raw = (WEIGHT_SUBSISTENANCE * subsistenace_normalised
                       + WEIGHT_DISCRETIONARY * discretionary_normalised
                       - WEIGHT_SLACK * slack)

                # squash into a range of -100 to 100 using a tanh function                
                commercial_demand_target = 100 * math.tanh(GAIN * raw)
            
            if commercial_demand_target < -100 or commercial_demand_target > 100:
                raise ValueError(f"Commercial demand target out of bounds: {commercial_demand_target}")
            print(f'COMMERCIAL TARGET: sub={monthly_subsistence_shortfall_units}, '
                  f'disc={monthly_unfilled_discretionary_demand}, sold={monthly_units_sold}')
            print(f'COMMERCIAL TARGET -> {commercial_demand_target:.2f}')
        
        commercial_demand_step = commercial_demand_target - commercial_demand
        commercial_demand_rise_rate = 1
        commercial_demand_fall_rate = 1

        # Phase 4: Adjust demand values towards their targets
        # Phase 4a: Residential demand        
        residential_demand += max(-residential_demand_fall_rate, min(residential_demand_rise_rate, residential_demand_step))
        if residential_demand < -100 or residential_demand > 100:
            raise ValueError(f"Residential demand out of bounds: {residential_demand}")
        
        # Phase 4b: Commercial demand (TODO)
        commercial_demand += max(-commercial_demand_fall_rate, min(commercial_demand_rise_rate, commercial_demand_step))
        if commercial_demand < -100 or commercial_demand > 100:
            raise ValueError(f"Commercial demand out of bounds: {commercial_demand}")

        # Phase 4c: Industrial demand (TODO)



        # Phase 5: Apply population changes (lots, births, deaths, migration)
        vacant_residential_lots = sum(1 for l in residential_lots_arr if not l.occupied)
        vacant_commercial_lots = sum(1 for l in commercial_lots_arr if not l.occupied)

        def calculate_births(residential_population: int, birth_rate) -> int:
            """
            Calculate the number of births in the city based on the residential population.

            Args:
                - residential_population (int): The current residential population of the city.
            Returns:
                - int: The number of births to add to the city.
            """
            births = max(math.floor(residential_population * birth_rate), 0)
            return births
        
        def calculate_deaths(residential_population: int, death_rate: float) -> int:
            """
            Calculate the number of deaths in the city based on the residential population.

            Args:
                - residential_population (int): The current residential population of the city.
                - death_rate (float): The death rate to apply to the population.
            Returns:
                - int: The number of deaths to remove from the city.
            """
            deaths = max(math.floor(residential_population * death_rate), 0)
            return deaths


        def calculate_migration(residential_demand: float, residential_population: int) -> int:
            """
            Calculate the number of migrants to add or remove from the city based on residential demand
            and unmet utility.

            Args:
                - residential_demand (float): The current residential demand, ranging from -100 to 100.
                - residential_population (int): The current residential population of the city.
                - 
            Returns:
                - int: The number of migrants to add (positive) or remove (negative) from the city.
            """
            if residential_demand < 0:
                migration = max(math.ceil(residential_population * 0.005), 0) * -1
            elif residential_demand > 0:
                # we need at least one migrant when demand is positive, otherwise
                # the city gets stuck at 0 population and can't recover.
                migration = max(math.ceil(residential_population * 0.005), 1)
            else:
                migration = 0
            return migration
        
        births = calculate_births(residential_population, CONF.GLOBAL_birth_rate)
        deaths = calculate_deaths(residential_population, CONF.GLOBAL_death_rate)
        migration = calculate_migration(residential_demand, residential_population)

        projected_resi_population_change = births - deaths + migration


        def adjust_vacant_resi_lots(vacant_residential_lots: int, residential_lots_arr: list[LotEntity]):
            """
            Adjust the number of vacant residential lots based on the current population and lot capacity.

            Args:
                - vacant_residential_lots (int): The current number of vacant residential lots.
                - residential_lots_arr (list[LotEntity]): The list of current residential lots.
            """
            if vacant_residential_lots < len(residential_lots_arr) * 0.05:
                # if vacant_residential_lots is less than 5% of lots, add some lots
                for _ in range(10):
                    if len(residential_lots_arr) >= CONF.GLOBAL_residential_lots_cap:
                        break
                    add_lot(lots, RciType.RESIDENTIAL, residential_lots_arr=residential_lots_arr)
            elif vacant_residential_lots > len(residential_lots_arr) * 0.5:
                # else if vacant_residential_lots is more than 50% of lots, remove some lots
                removed = 0
                for lot in [l for l in residential_lots_arr if not l.occupied]:
                    if removed >=10 or len(residential_lots_arr) <= 100:
                        break
                    residential_lots_arr.remove(lot)
                    lots.remove(lot)
                    removed += 1
        
        
        def adjust_vacant_commercial_lots(vacant_commercial_lots: int, commercial_lots_arr: list[LotEntity]):
            """
            Adjust the number of vacant commercial lots based on the current population and lot capacity.

            Args:
                - vacant_commercial_lots (int): The current number of vacant commercial lots.
                - commercial_lots_arr (list[LotEntity]): The list of current commercial lots.
            """
            if vacant_commercial_lots < len(commercial_lots_arr) * 0.05:
                # if vacant_commercial_lots is less than 5% of lots, add some lots
                for _ in range(10):
                    if len(commercial_lots_arr) >= CONF.GLOBAL_commercial_lots_cap:
                        break
                    add_lot(lots, RciType.COMMERCIAL, commercial_lots_arr=commercial_lots_arr)
            elif vacant_commercial_lots > len(commercial_lots_arr) * 0.5:
                # else if vacant_commercial_lots is more than 50% of lots, remove some lots
                removed = 0
                for lot in [l for l in commercial_lots_arr if not l.occupied]:
                    if removed >=10 or len(commercial_lots_arr) <= 100:
                        break
                    commercial_lots_arr.remove(lot)
                    lots.remove(lot)
                    removed += 1

        # TODO, this `vacant_residential_lots` check stuff can be moved off somewhere else in the routine
        if is_week_end:
            adjust_vacant_resi_lots(vacant_residential_lots, residential_lots_arr)

            adjust_vacant_commercial_lots(vacant_commercial_lots, commercial_lots_arr)


        if (residential_population + projected_resi_population_change) > len(residential_lots_arr):
            print(f"WARN: Cannot move residential population from {residential_population} to {residential_population + projected_resi_population_change} with only {len(residential_lots_arr)} lots")

        if projected_resi_population_change > 0:
            for _ in range(projected_resi_population_change):
                # add_lot
                # if not any (not l.occupied and l.rci_type == RciType.RESIDENTIAL for l in lots):
                    # pass
                    #add_lot(RciType.RESIDENTIAL)
                # if there are any vacant residential lots, add a resident to the city
                if any (not l.occupied and l.rci_type == RciType.RESIDENTIAL for l in lots):
                    if add_entity_to_city(lots, RciType.RESIDENTIAL, CONF):
                        residential_population += 1
        elif projected_resi_population_change < 0:
            # if more people are leaving (or dying) than arriving,
            # pick a resident to remove from the city, and free up the lot
            # they were occupying. TODO: A more sophisticated approach would be
            # to remove a specific resident. At the moment, it just removes the
            # last resident in the residential_entities list. Fine for the sim
            # just now, but might have consequences once residents are employed
            # and have specific jobs.
            for _ in range(abs(projected_resi_population_change)):
                if residential_population > 0:
                    # remove a resident from the city
                    residential_population -= 1
                    # find a residential entity to remove
                    if residential_entities:
                        entity_to_remove = residential_entities.pop()
                        # free up the lot it was occupying
                        lot_to_free = next(
                            (l for l in lots
                             if l.occupied and l.occupied_by == entity_to_remove.id
                             and l.rci_type == RciType.RESIDENTIAL),
                            None,
                        )
                        if lot_to_free:
                            lot_to_free.occupied = False
                            lot_to_free.occupied_by = None

        def calculate_open_close_commercial(commercial_demand: float, commercial_population: int) -> int:
            """
            Calculate how many commercial entities will open or close based on the commercial demand and current commercial pop.

            Args:
                - commercial_demand (float): The current commercial demand, ranging from -100 to 100.
                - commercial_population (int): The current commercial population of the city.
            Returns:
                - int: The number of commercial entities to open (positive) or close (negative).
            See also:
                - calculate_open_close_industrial (industrial)
                - calculate_migration (residential)
            """
            if commercial_demand < 0:
                projected_change = max(math.ceil(commercial_population * 0.005), 0) * -1 # TODO: move 0.005 to config, it represents the rate of commercial population change per day
            elif commercial_demand > 0:
                # Ensure at least one commercial entity opens if there is positive demand (otherwise
                # the economy gets stuck in a rut if it ever hits 0)
                projected_change = max(math.ceil(commercial_population * 0.005), 1) # TODO: move 0.005 to config
            else:
                projected_change = 0
            return projected_change

        projected_commercial_population_change = calculate_open_close_commercial(commercial_demand, len(commercial_entities))
        if projected_commercial_population_change + len(commercial_entities) > len(commercial_lots_arr):
            print(f"WARN: Cannot move commercial population from {len(commercial_entities)} to {projected_commercial_population_change + len(commercial_entities)} with only {len(commercial_lots_arr)} lots")

        if projected_commercial_population_change > 0:
            for _ in range(projected_commercial_population_change):
                if any(not l.occupied and l.rci_type == RciType.COMMERCIAL for l in lots):
                    add_entity_to_city(lots, RciType.COMMERCIAL, CONF)
        elif projected_commercial_population_change < 0:
            for _ in range(abs(projected_commercial_population_change)):
                if not commercial_entities:
                    break
                remove_employer_entity(commercial_entities[-1], commercial_entities, reason='demand')


        # Phase 6: Resolve employment
        def employ_residents(residential_entities: list[ResidentialEntity],
                             commercial_entities: list[CommercialEntity],
                             industrial_entities: list[IndustrialEntity]) -> None:
            """
            Assign residents to available jobs in Commercial and Industrial entities.


            Args:
                - residential_entities (list[ResidentialEntity]): The list of residential entities (residents)
                - commercial_entities (list[CommercialEntity]): The list of commercial entities (businesses)
                - industrial_entities (list[IndustrialEntity]): The list of industrial entities (factories/wholesalers)
            Returns:
                None
            """
            # Reset employment status for all residents
            for resident in residential_entities:
                resident.employed = False
                resident.employed_at = None
                resident.employed_type = None

            idx = 0
            for commercial in commercial_entities:
                for _ in range(commercial.jobs):
                    if idx < len(residential_entities):
                        resident = residential_entities[idx]
                        resident.employed = True
                        resident.employed_at = commercial.id
                        resident.employed_type = RciType.COMMERCIAL
                        idx += 1
            for industrial in industrial_entities:
                for _ in range(industrial.jobs):
                    if idx < len(residential_entities):
                        resident = residential_entities[idx]
                        resident.employed = True
                        resident.employed_at = industrial.id
                        resident.employed_type = RciType.INDUSTRIAL
                        idx += 1

        employ_residents(residential_entities, commercial_entities, industrial_entities)


        def pay_employees(bank: Bank,
                            residential_entities: list[ResidentialEntity],
                            commercial_entities: list[CommercialEntity],
                            industrial_entities: list[IndustrialEntity],
                            tax_rate_residential: float) -> None:
                """
                Pay employees and collect taxes to the city coffers.
    
                Args:
                    - bank (Bank): The bank instance to handle transactions.
                    - residential_entities (list[ResidentialEntity]): The list of residential entities (residents)
                    - commercial_entities (list[CommercialEntity]): The list of commercial entities (businesses)
                    - industrial_entities (list[IndustrialEntity]): The list of industrial entities (factories/wholesalers)
                    - tax_rate_residential (float): The tax rate for residential entities.
                Returns:
                    None
                """
                commercial_by_id = {c.id: c for c in commercial_entities}
                industrial_by_id = {i.id: i for i in industrial_entities}

                for resident in residential_entities:
                    if resident.employed:
                        employer = None
                        if resident.employed_type == RciType.COMMERCIAL:
                            employer = commercial_by_id.get(resident.employed_at)
                        elif resident.employed_type == RciType.INDUSTRIAL:
                            employer = industrial_by_id.get(resident.employed_at)
                        if employer:
                            salary = employer.employee_wage
                            tax = salary * tax_rate_residential
                            net_salary = salary - tax
                            if bank.get_balance(employer.id_uuid) < Decimal(salary):
                                raise NotEnoughMoneyError(f'Employer {employer.name} (UUID: {employer.id_uuid}) does not have enough money to pay salary of {salary}. Current balance: {bank.get_balance(employer.id_uuid)}')
                            try:
                                # Transfer the net salary to the resident
                                bank.transfer_money(sender=employer.id_uuid, receiver=resident.id_uuid, amount=Decimal(net_salary), for_good_service="employee_salary")
                                # Transfer the tax to the city coffers (resident income tax follows UK style PAYE system)
                                bank.transfer_money(sender=employer.id_uuid, receiver=the_city.id_uuid, amount=Decimal(tax), for_good_service="employee_income_tax")
                            except NotEnoughMoneyError as e:
                                print(f"Error: during salary payment: {e}")
                                raise e



        # Phase 7: Settle economics (wages, taxation, consumption, P&L, bankruptcy)
        # TODO: once consumption is implemented, remove the reset balance function

        # def temp_reset_industrial_stock(industrial_entities: list[IndustrialEntity]) -> None:
        #     """
        #     Reset the stock of all industrial entities to their starting stock capacity.
        #     """
        #     print('--- TEMP: Resetting industrial stock to capacity ---')
        #     for i in industrial_entities:
        #         i.stock += i.stock_capacity - i.stock
        # temp_reset_industrial_stock(industrial_entities)

        def commercial_purchase_from_industrial(bank: Bank,
                                                commercial_entities: list[CommercialEntity],
                                                industrial_entities: list[IndustrialEntity],
                                                conf: GlobalConfig) -> None:
            """
            Simulate commercial entities purchasing goods from industrial entities.

            Each commercial entity will attempt to purchase goods from industrial entities
            to fill their stock up to their stock capacity. The amount purchased is deducted
            from the industrial entity's stock, and added to the commercial entity's stock.
            The commercial entity's bank balance is reduced by the cost of goods purchased,
            and the industrial entity's bank balance is increased by the same amount. Sales tax is
            collected and added to the city coffers.
            """

            def get_industrial_entities_with_stock(industrial_entities: list[IndustrialEntity]) -> list[IndustrialEntity]:
                """
                Get a list of industrial entities that have stock available for sale.
                """
                return [i for i in industrial_entities if i.stock > 0]
        
            print('--- Commercial entities purchasing goods from industrial entities ---')
            
            def get_commercial_entities_needing_stock(commercial_entities: list[CommercialEntity]) -> list[CommercialEntity]:
                """
                Get a list of commercial entities that need to purchase stock,
                sorted by their current stock level (lowest first).
                """
                return sorted(
                    (c for c in commercial_entities if c.stock < c.stock_capacity),
                    key=lambda c: c.stock
                )
            
            def buy_one_stock_item(commercial_entity: CommercialEntity, industrial_entity: IndustrialEntity) -> bool:
                """
                Attempt to buy a single unit of stock from an industrial entity.
                
                Args:
                    - commercial_entity (CommercialEntity): The commercial entity attempting to buy.
                    - industrial_entity (IndustrialEntity): The industrial entity selling.
                Returns:
                    - bool: True if the purchase was successful, False otherwise.
                """
                price = Decimal(industrial_entity.stock_price_per_unit)
                if bank.get_balance(commercial_entity.id_uuid) < price:
                    print(f"Commercial entity {commercial_entity.name} (UUID: {commercial_entity.id_uuid}) does not have enough money to buy from {industrial_entity.name} (UUID: {industrial_entity.id_uuid}). Commercial balance: {bank.get_balance(commercial_entity.id_uuid)}, Price: {price}")
                    return False
                try:
                    bank.transfer_money(
                        sender=commercial_entity.id_uuid,
                        receiver=industrial_entity.id_uuid,
                        amount=price,
                        for_good_service="good"
                    )
                except NotEnoughMoneyError as e:
                    print(f"Error: during commercial purchase transaction: {e}")
                    raise e
                industrial_entity.stock -= 1
                commercial_entity.stock += 1
                return True
            
            commercial_subsistence_unmet = 0
            units_sold_ind_to_comm = 0
            commercial_entities_needing_stock = get_commercial_entities_needing_stock(commercial_entities)
            industrial_entities_with_stock = get_industrial_entities_with_stock(industrial_entities)
            while commercial_entities_needing_stock:
                def get_vendors_sorted_by_price(industrial_entities_with_stock):
                    return sorted(industrial_entities_with_stock, key=lambda i: i.stock_price_per_unit)

                vendors = get_vendors_sorted_by_price(industrial_entities_with_stock)
                if not vendors:
                    print(f'There were {len(commercial_entities_needing_stock)} commercial entities needing stock, but no industrial entities had stock available.')
                    break
                commercial_still_needing_stock = []
                for c in commercial_entities_needing_stock:
                    if c.stock < c.stock_capacity:
                        cheapest_vendor = vendors[0] if vendors else None
                        if cheapest_vendor is None:
                            print(f"Commercial entity {c.name} (UUID: {c.id_uuid}) could not find any industrial entities with stock to purchase.")
                            break
                        if buy_one_stock_item(c, cheapest_vendor):
                            units_sold_ind_to_comm += 1
                            if c.stock < c.stock_capacity:
                                commercial_still_needing_stock.append(c)
                        else:
                            print(f"Commercial entity {c.name} (UUID: {c.id_uuid}) could not afford to buy from {cheapest_vendor.name} (UUID: {cheapest_vendor.id_uuid}).")
                        industrial_entities_with_stock = get_industrial_entities_with_stock(industrial_entities)
                commercial_entities_needing_stock = [c for c in commercial_still_needing_stock if get_industrial_entities_with_stock(industrial_entities)]

            # record subsistence outcome for commercial entities
            for c in commercial_entities:
                c.subsistence_q_met = c.stock >= c.stock_capacity
                if not c.subsistence_q_met:
                    commercial_subsistence_unmet += (c.stock_capacity - c.stock)


            return {
                'units_sold_ind_to_comm': units_sold_ind_to_comm,
                'commercial_subsistence_unmet': commercial_subsistence_unmet
            }


        def industrial_purchase_from_wholesaler(bank: Bank,
                                                industrial_entities: list[IndustrialEntity],
                                                the_wholesaler: WholesalerEntity,
                                                conf: GlobalConfig) -> None:
            """
            Simulate industrial entities purchasing goods from the global wholesaler.

            Each industrial entity will attempt to purchase goods from the wholesaler
            to meet their stock requirements. The amount purchased is deducted from
            the wholesaler's stock and added to the industrial entity's stock. The
            industrial entity's bank balance is reduced by the cost of goods purchased,
            and the wholesaler's bank balance is increased by the same amount.

            Other purchase methods perform round-robin type purchasing. However, since the
            wholesaler is a single entity, the industrial entities will purchase from it
            in bulk, depending on their stock requirements. The wholesaler has an unlimited
            stock capacity, so is always able to meet the industrial entities' demands.

            The industrial entity's purchase does not diminish the wholesaler's stock, as it
            is an unlimited stock source.
            """
            print('--- Industrial entities purchasing goods from wholesaler ---')
            for industrial in industrial_entities:
                # choose how much stock to purchase based on the entity's stock capacity
                # and current balance. Leave some buffer in the balance to avoid bankrupting the entity.
                stock_needed = industrial.stock_capacity - industrial.stock
                if stock_needed <= 0:
                    continue
                price_per_unit = Decimal(the_wholesaler.stock_price_per_unit)
                max_affordable_units = int(bank.get_balance(industrial.id_uuid) // price_per_unit)
                units_to_purchase = min(stock_needed, max_affordable_units)
                if units_to_purchase <= 0:
                    print(f"Industrial entity {industrial.name} (UUID: {industrial.id_uuid}) cannot afford to purchase any stock from wholesaler. Balance: {bank.get_balance(industrial.id_uuid)}, Price per unit: {price_per_unit}")
                    continue
                total_cost = units_to_purchase * price_per_unit
                try:
                    bank.transfer_money(
                        sender=industrial.id_uuid,
                        receiver=the_wholesaler.id_uuid,
                        amount=total_cost,
                        for_good_service="good"
                    )
                except NotEnoughMoneyError as e:
                    print(f"Error: during industrial purchase transaction: {e}")
                    raise e
                industrial.stock += units_to_purchase
                print(f"Industrial entity {industrial.name} (UUID: {industrial.id_uuid}) purchased {units_to_purchase} units from wholesaler for a total cost of {total_cost}. New stock: {industrial.stock}, New balance: {bank.get_balance(industrial.id_uuid)}")


        def resident_consume_from_commercial(
                bank: Bank,
                residential_entities: list[ResidentialEntity],
                commercial_entities: list[CommercialEntity],
                conf: GlobalConfig,
        ):
            """
            Simulate residents consuming goods from commercial entities.

            Each employed resident maximised their utility by consuming goods from 
            commercial entities. The amount consumed is deducted from the commercial
            entity's stock, and added to the resident's stock. The resident's bank
            balance is reduced by the cost of goods consumed, and the commercial
            entity's bank balanced is increased by the same amount. Sales tax is
            collected and added to the city coffers.
            """
            print('--- Residents consuming goods from commercial entities ---')
            subsistence_q = conf.GLOBAL_residential_subsistence_quantity

            for r in residential_entities:
                r.basket_count = 0
                r.period_utility = Decimal(0)
                r.subsistence_q_met = False
            
            def sellers_with_stock() -> list[CommercialEntity]:
                """
                Get a sorted list of commercial entities that have stock available for sale.
                """
                return sorted(
                    (c for c in commercial_entities if c.stock > 0),
                    key=lambda c: c.stock_price_per_unit
                )
            
            def buy_one_item(resident_entity, commercial_entity) -> bool:
                """
                Attempt to buy a single unit of stock from a commercial entity
                
                Args:
                    - resident_entity (ResidentialEntity): The resident attempting to buy.
                    - commercial_entity (CommercialEntity): The commercial entity selling.
                Returns:
                    - bool: True if the purchase was successful, False otherwise.
                """
                price = Decimal(commercial_entity.stock_price_per_unit)
                if bank.get_balance(resident_entity.id_uuid) < price:
                    print(f"Resident {resident_entity.name} (UUID: {resident_entity.id_uuid}) does not have enough money to buy from {commercial_entity.name} (UUID: {commercial_entity.id_uuid}). Resident balance: {bank.get_balance(resident_entity.id_uuid)}, Price: {price}")
                    return False
                try:
                    bank.transfer_money(
                        sender=resident_entity.id_uuid,
                        receiver=commercial_entity.id_uuid,
                        amount=price,
                        for_good_service="good"
                    )
                except NotEnoughMoneyError as e:
                    print(f"Error: during consumption transaction: {e}")
                    raise e
                commercial_entity.stock -= 1
                resident_entity.period_utility += resident_entity.marginal_utility(resident_entity.basket_count)
                resident_entity.basket_count += 1
                return True
            
            units_sold = 0

            # first pass: meet subsistence quantity for all residents
            residents_needing_subsistence = [r for r in residential_entities if subsistence_q > 0]
            while residents_needing_subsistence:
                sellers = sellers_with_stock()
                if not sellers:
                    print(f'There were {len(residents_needing_subsistence)} residents needing subsistence, but no commercial entities had stock available.')
                    break
                residents_still_needing_subsistence = []
                for r in residents_needing_subsistence:
                    if r.basket_count >= subsistence_q:
                        continue
                    cheapest = sellers[0] if sellers else None
                    if cheapest is None:
                        print(f"Resident {r.name} (UUID: {r.id_uuid}) could not find any commercial entities with stock to meet subsistence needs.")
                        break
                    if buy_one_item(r, cheapest):
                        units_sold += 1
                        if r.basket_count < subsistence_q:
                            residents_still_needing_subsistence.append(r)
                    else:
                        print(f"Resident {r.name} (UUID: {r.id_uuid}) could not afford to buy from {cheapest.name} (UUID: {cheapest.id_uuid}).")
                    sellers = sellers_with_stock()
                residents_needing_subsistence = [r for r in residents_still_needing_subsistence if sellers_with_stock()]

            ## record the subsistence outcome
            resident_subsistence_unmet = []
            subsistence_shortfall_units = 0
            for r in residential_entities:
                r.subsistence_q_met = r.basket_count >= subsistence_q
                if not r.subsistence_q_met:
                    subsistence_shortfall_units += (subsistence_q - r.basket_count)
                    resident_subsistence_unmet.append(r)

            # second pass: allow residents to take part in discretionary consumption
            unfilled_discretionary_demand = 0
            active_residents = list(residential_entities)
            while active_residents:
                sellers = sellers_with_stock()
                if not sellers:
                    # There's no stock left anywhere, so any resident who would buy at the cheapest price
                    # they last saw and can afford it should be counted towards unfilled_discretionary_demand.
                    if commercial_entities:
                        cheapest_price = min(c.stock_price_per_unit for c in commercial_entities)
                        for r in active_residents:
                            if (r.willing_to_buy(cheapest_price) and bank.get_balance(r.id_uuid) >= cheapest_price):
                                unfilled_discretionary_demand += 1
                    print(f'There were {len(active_residents)} active residents wanting discretionary consumption, but no commercial entities had stock available, creating {unfilled_discretionary_demand} unfilled discretionary demand.')
                    break
                residents_to_drop = []
                for r in active_residents:
                    choice = None
                    for c in sellers:
                        if c.stock <= 0:
                            # TODO: should we remove this from the list here?
                            continue
                        price = Decimal(c.stock_price_per_unit)
                        if r.willing_to_buy(price) and bank.get_balance(r.id_uuid) >= price:
                            choice = c
                            break
                    if choice is None:
                        # the resident isn't willing to buy any more
                        residents_to_drop.append(r)
                        continue
                    if buy_one_item(r, choice):
                        units_sold += 1
                    else:
                        residents_to_drop.append(r)
                    sellers = sellers_with_stock()
                for r in residents_to_drop:
                    active_residents.remove(r)
            
            # commit resident period utility to lifetime utility
            for r in residential_entities:
                r.lifetime_utility += r.period_utility
            
            return {
                'resident_subsistence_unmet': resident_subsistence_unmet,
                'subsistence_shortfall_units': subsistence_shortfall_units,
                'units_sold': units_sold,
                'unfilled_discretionary_demand': unfilled_discretionary_demand

            }

        
        def bankrupt_entities(bank: Bank,
                              employing_entities: list[CommercialEntity] | list[IndustrialEntity],
                              lowest_goods_price: Decimal = Decimal(0)) -> list:
            """
            Check for bankrupt employing entities, and remove them from the city.

            The `employing_entities` param can be either `CommercialEntity` or `IndustrialEntity`, but cannot be mixed.

            Args:
                - bank (Bank): The bank instance to handle transactions.
                - employing_entities (list[CommercialEntity] | list[IndustrialEntity]): The list of employing entities (businesses)
                - lowest_goods_price (Decimal): The lowest goods price for reference (not required for commercial entities)
            Returns:
                - list: A report of bankrupt entities, including the reason for their bankruptcy.
            Raises:
                - ValueError: If employing_entities neither RciType.COMMERCIAL nor RciType.INDUSTRIAL
                - ValueError: If the wholesaler is not provided when bankrupting entities of type RciType.INDUSTRIAL
            """
            report = []
            for employer_entity in list(employing_entities):
                balance = bank.get_balance(employer_entity.id_uuid)

                def employees_of(employer: CommercialEntity | IndustrialEntity) -> list[ResidentialEntity]:
                    """Return the residents who are employed by the given employer."""
                    return [
                        r for r in residential_entities
                        if r.employed
                        and r.employed_at == employer.id
                        and r.employed_type == employer.rci_type
                    ]
                def count_employees_of(employer: CommercialEntity | IndustrialEntity) -> int:
                    """Return the number of residents who are employed by the given employer."""
                    return len(employees_of(employer))

                # Cannot cover wage obligations (num jobs * wage)
                employee_obligations = Decimal(count_employees_of(employer_entity) * employer_entity.employee_wage)
                cannot_pay_wages = balance < employee_obligations

                # Holds no stock, and cannot afford the cheapest restock
                cannot_restock = employer_entity.stock <= 0 and balance < Decimal(lowest_goods_price)

                if cannot_pay_wages or cannot_restock:
                    reason = 'bankrupt: wages' if cannot_pay_wages else 'bankrupt: restock'
                    print(f"Employer entity {employer_entity.name} ({employer_entity.rci_type}) "
                          f"(UUID: {employer_entity.id_uuid}) is bankrupt ({reason}) and will be removed from the city.")
                    report.append(remove_employer_entity(employer_entity, employing_entities, reason))
            return report


        def temp_reset_industrial_employer_balance(bank: Bank,
                                        industrial_entities: list[IndustrialEntity],
                                        conf: GlobalConfig) -> None:
            """
            Reset the balance of all industrial entities to their starting balance

            TODO: Remove this function once the simulation can handle bankruptcy and running out of money.


            Args:
                - bank (Bank): The bank instance to handle transactions.
            """
            for industrial in industrial_entities:
                balance = bank.get_balance(industrial.id_uuid)
                if balance < Decimal(conf.GLOBAL_industrial_starting_money):
                    bank_account = bank.get_account_by_owner(industrial.id_uuid)
                    bank_account.balance = Decimal(conf.GLOBAL_industrial_starting_money)

        industrial_purchase_report = industrial_purchase_from_wholesaler(the_bank, industrial_entities, the_wholesaler, CONF)
        commercial_consumption_report = commercial_purchase_from_industrial(the_bank, commercial_entities, industrial_entities, CONF)
        consumption_report = resident_consume_from_commercial(bank=the_bank, residential_entities=residential_entities, commercial_entities=commercial_entities, conf=CONF)
        # temp_reset_industrial_employer_balance(bank=the_bank, industrial_entities=industrial_entities, conf=CONF)
        commercial_bankruptcy_report = bankrupt_entities(bank=the_bank, employing_entities=commercial_entities, lowest_goods_price=min((i.stock_price_per_unit for i in industrial_entities), default=Decimal(0)))   
        industrial_bankruptcy_report = bankrupt_entities(bank=the_bank, employing_entities=industrial_entities, lowest_goods_price=Decimal(the_wholesaler.stock_price_per_unit))   
        pay_employees(the_bank, residential_entities, commercial_entities, industrial_entities, tax_rate_residential)

        print(f'City bank balance: {the_bank.get_balance(the_city.id_uuid):.2f}')
        print(f'Whole economy money: {the_bank.sum_all_accounts():.2f}')
        print(f'Total money in industrial: {the_bank.sum_all_accounts(entity_type="IndustrialEntity"):.2f}')
        print(f'Total money in commercial: {the_bank.sum_all_accounts(entity_type="CommercialEntity"):.2f}')
        print(f'Total money in residential: {the_bank.sum_all_accounts(entity_type="ResidentialEntity"):.2f}')
        print(f'Total money in wholesaler: {the_bank.sum_all_accounts(entity_type="WholesalerEntity"):.2f}')
        print(f'Total money in government: {the_bank.sum_all_accounts(entity_type="CityEntity"):.2f}')

        print(f'Number of residents employed in commercial: {sum(1 for r in residential_entities if r.employed and r.employed_type == RciType.COMMERCIAL)}')
        print(f'Number of residents employed in industrial: {sum(1 for r in residential_entities if r.employed and r.employed_type == RciType.INDUSTRIAL)}')

        # Phase 8: Log history
        history['residential_population'].append(len(residential_entities))
        history['commercial_population'].append(len(commercial_entities))
        history['industrial_population'].append(len(industrial_entities))
        history['residential_demand'].append(residential_demand)
        history['commercial_demand'].append(commercial_demand)
        history['industrial_demand'].append(industrial_demand)
        history['residential_demand_target'].append(residential_demand_target)
        history['commercial_demand_target'].append(commercial_demand_target)
        history['industrial_demand_target'].append(industrial_demand_target)
        history['residential_lots'].append(len(residential_lots_arr))
        history['commercial_lots'].append(len(commercial_lots_arr))
        history['industrial_lots'].append(len(industrial_lots_arr))
        history['jobs_per_resident'].append(total_jobs / len(residential_entities) if len(residential_entities) > 0 else 0)
        history['migration'].append(migration)
        history['deaths'].append(deaths)
        history['births'].append(births)
        history['vacancy_rate'].append(vacancy_rate)
        history['unemployment_rate'].append(unemployment_rate)

        history['resident_subsistence_unmet'].append(consumption_report['resident_subsistence_unmet']) # not sure what I'm trying to store here, this is a list
        history['subsistence_shortfall_units'].append(consumption_report['subsistence_shortfall_units'])
        history['units_sold'].append(consumption_report['units_sold'])
        history['unfilled_discretionary_demand'].append(consumption_report['unfilled_discretionary_demand'])
        history['commercial_subsistence_unmet'].append(commercial_consumption_report['commercial_subsistence_unmet'])
        history['units_sold_ind_to_comm'].append(commercial_consumption_report['units_sold_ind_to_comm'])

        history['total_money_in_economy'].append(the_bank.sum_all_accounts())
        history['total_money_in_industrial'].append(the_bank.sum_all_accounts(entity_type='IndustrialEntity'))
        history['total_money_in_commercial'].append(the_bank.sum_all_accounts(entity_type='CommercialEntity'))
        history['total_money_in_residential'].append(the_bank.sum_all_accounts(entity_type='ResidentialEntity'))
        history['total_money_in_wholesale'].append(the_bank.sum_all_accounts(entity_type='WholesalerEntity'))
        history['total_money_in_government'].append(the_bank.sum_all_accounts(entity_type='CityEntity'))

    def plot_history():
        # fig_combined = plot_demand_history(
        #     residential_demand_history=history['residential_demand'],
        #     commercial_demand_history=history['commercial_demand'],
        #     industrial_demand_history=history['industrial_demand'],
        #     title="RCI Demand Over Time",
        # )
        # display(fig_combined)

        fig = plot_demand_detail(
            demand_history=history['residential_demand'],
            demand_target_history=history['residential_demand_target'],
            population_history=history['residential_population'],
            migration_history=history['migration'],
            births_history=history['births'],
            deaths_history=history['deaths'],
            lots_history=history['residential_lots'],
            title="Residential Detail",
            demand_color="rgb(76, 175, 80)",  # or use RESIDENTIAL_COLOR
        )
        display(fig)

        fig_res = plot_demand_history(
            residential_demand_history=history['residential_demand'],
            residential_target_history=history['residential_demand_target'],
            population_history=history['residential_population'],
            population_label="Residential Population",
            title="Residential Demand Over Time",
        )
        display(fig_res)

        fig_com = plot_demand_history(
            commercial_demand_history=history['commercial_demand'],
            commercial_target_history=history['commercial_demand_target'],
            population_history=history['commercial_population'],
            population_label="Commercial Population",
            title="Commercial Demand Over Time",
        )
        display(fig_com)

        fig_ind = plot_demand_history(
            industrial_demand_history=history['industrial_demand'],
            industrial_target_history=history['industrial_demand_target'],
            population_history=history['industrial_population'],
            population_label="Industrial Population",
            title="Industrial Demand Over Time",
        )
        display(fig_ind)

        fig_jobs = plot_jobs_history(
            jobs_per_resident_history=history['jobs_per_resident'],
            unemployment_rate_history=history['unemployment_rate'],
            vacancy_rate_history=history['vacancy_rate'],
            title="Jobs and Employment Over Time",
        )
        display(fig_jobs)

        fig_welfare = plot_welfare_history(
            subsistence_unmet_count_history=[len(u) for u in history['resident_subsistence_unmet']],
            subsistence_shortfall_units_history=history['subsistence_shortfall_units'],
            title="Welfare Over Time",
        )
        display(fig_welfare)

        fig_throughput = plot_throughput_history(
            units_sold_history=history['units_sold'],
            population_history=history['residential_population'],
            subsistence_quantity=CONF.GLOBAL_residential_subsistence_quantity,
            title="Throughput Over Time",
        )
        display(fig_throughput)

        fig_shortage = plot_shortage_vs_demand_target(
            unfilled_discretionary_demand_history=history['unfilled_discretionary_demand'],
            commercial_demand_target_history=history['commercial_demand_target'],
            title="Shortage vs Commercial Demand Target",
        )
        display(fig_shortage)

    
    plot_history()



main(20)

Creating account for owner RestOfWorldEntity_0 (UUID: 11f18863-5bf0-4a0b-8fc0-b9aedbbe2438, type RestOfWorldEntity)
Creating account for owner City (UUID: 49afb744-a703-4350-8d93-b4dd88547eac, type CityEntity)
Creating account for owner WholesalerEntity_0 (UUID: 5f1b1910-0213-4d99-a24e-fa5caa1f7c83, type WholesalerEntity)
Added residential lot (id: 0)
Added residential lot (id: 1)
Added residential lot (id: 2)
Added residential lot (id: 3)
Added residential lot (id: 4)
Added residential lot (id: 5)
Added residential lot (id: 6)
Added residential lot (id: 7)
Added residential lot (id: 8)
Added residential lot (id: 9)
Added residential lot (id: 10)
Added residential lot (id: 11)
Added residential lot (id: 12)
Added residential lot (id: 13)
Added residential lot (id: 14)
Added residential lot (id: 15)
Added residential lot (id: 16)
Added residential lot (id: 17)
Added residential lot (id: 18)
Added residential lot (id: 19)
Added residential lot (id: 20)
Added residential lot (id: 21)
Adde